# 01 — Data Loading & Cleaning

This notebook loads the supply-chain fact data, performs basic data-quality checks, applies the provided cleaning rule for unknown customer demographics, calculates Gross Sales, and checks for price outliers.

> **Note:** The customer-demographic replacement (`Unknown` → `Male`) is kept exactly as provided in the supplied analysis.


## 1. Import libraries

`pandas` is used for tabular data analysis and `numpy` is imported for numerical operations.


In [ ]:
import pandas as pd
import numpy as np


## 2. Load the dataset

The project already contains the source fact table under `data/raw/`. Using a relative path keeps the notebook reproducible when opened from the `python/` folder.


In [ ]:
df = pd.read_csv("../data/raw/Fact_Orders.csv")


## 3. Initial inspection

These checks show sample records, the dataset dimensions, column data types, and the structure of the data.


In [ ]:
df.head()


In [ ]:
df.shape


In [ ]:
df.info()


## 4. Missing-value and duplicate checks

First, inspect missing values and duplicate rows so data-quality issues can be identified before analysis.


In [ ]:
df.isnull()


In [ ]:
print("Number of duplicate rows:")
df.duplicated().sum()


In [ ]:
df.isnull().sum()


In [ ]:
df.duplicated()


## 5. Remove duplicate rows

The duplicate-removal step follows the supplied analysis. `drop_duplicates()` returns the cleaned table; assigning it back to `df` makes the cleaning permanent for the remaining notebook steps.


In [ ]:
df = df.drop_duplicates()


In [ ]:
df.isnull().sum()


## 6. Basic validity checks

Check for non-positive order quantities and prices, then sort by price to inspect unusually low or high values.


In [ ]:
print("Non-positive order quantities:", (df["Order quantities"] <= 0).sum())
print("Non-positive prices:", (df["Price"] <= 0).sum())


In [ ]:
df.sort_values("Price")


## 7. Descriptive statistics

`describe()` summarizes numerical columns, while `describe(include="object")` summarizes categorical/text columns.


In [ ]:
print("Descriptive statistics:")
df.describe()


In [ ]:
print("Categorical descriptive statistics:")
df.describe(include="object")


## 8. Clean customer demographics

The supplied analysis replaces `Unknown` with `Male`. This transformation is retained here so the project matches the submitted logic.


In [ ]:
df["Customer demographics"] = df["Customer demographics"].replace("Unknown", "Male")


In [ ]:
print("Categorical descriptive statistics after demographic cleaning:")
df.describe(include="object")


## 9. Calculate Gross Sales

Gross Sales are calculated as order quantity multiplied by unit price.


In [ ]:
df["Gross_Sales"] = (
    df["Order quantities"] *
    df["Price"]
)


In [ ]:
df.head()


## 10. Price outlier detection

The IQR method identifies prices below the lower bound or above the upper bound. These records are flagged for review rather than automatically deleted.


In [ ]:
# Calculate the first quartile (Q1)
Q1 = df["Price"].quantile(0.25)

# Calculate the third quartile (Q3)
Q3 = df["Price"].quantile(0.75)

# Calculate the Interquartile Range (IQR)
IQR = Q3 - Q1

# Calculate the lower and upper limits
lower = Q1 - (1.5 * IQR)
upper = Q3 + (1.5 * IQR)

# Find outliers
outliers = df[
    (df["Price"] < lower) |
    (df["Price"] > upper)
]

# Display the number of outliers
print("Number of Outliers:", len(outliers))

# Display the outlier rows
outliers


## 11. Save the cleaned fact table

Save the cleaned and enriched fact data so the second notebook can continue from the processed dataset.


In [ ]:
df.to_csv("../data/processed/Fact_Orders_enriched.csv", index=False)
print("Cleaned fact table saved successfully.")


## Cleaning stage complete

At this point, the fact data has been inspected, duplicates handled, the provided demographic rule applied, Gross Sales calculated, and price outliers identified for review.
